In [0]:
%pip install torch transformers

In [0]:
import torch
from typing import Iterator
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, DateType
import pandas as pd
from delta.tables import DeltaTable
import os
import shutil

# ==========================================
# CONSTANTS & CONFIGURATIONS
# ==========================================
prices_table_name = "finance_intelligence_hub.bronze.stock_prices_raw"
news_table_name = "finance_intelligence_hub.bronze.company_news_polygon"
temp_results_table = "finance_intelligence_hub.bronze.temp_finbert_results"

catalog_name = "finance_intelligence_hub"
schema_name = "bronze"
volume_name = "finbert_model_volume"
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

print("🛡️ Ensuring Unity Catalog Volume exists for Serverless Compute...")
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
except Exception as e:
    print(f"ℹ️ Volume storage verified or managed externally: {e}")

shared_model_dir = os.path.join(volume_path, "finbert_core")
model_udf_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/finbert_core"


def _model_is_complete(model_dir: str) -> bool:
    if not os.path.exists(os.path.join(model_dir, "config.json")):
        return False
    has_weights = (
        os.path.exists(os.path.join(model_dir, "model.safetensors"))
        or os.path.exists(os.path.join(model_dir, "pytorch_model.bin"))
    )
    return has_weights


if not _model_is_complete(shared_model_dir):
    print("📥 Model missing or incomplete on the Volume. (Re)downloading FinBERT...")
    if os.path.exists(shared_model_dir):
        print("🧹 Removing incomplete previous copy from the Volume...")
        shutil.rmtree(shared_model_dir)

    local_driver_dir = "/tmp/finbert_local_driver_download"
    if os.path.exists(local_driver_dir):
        shutil.rmtree(local_driver_dir)
    os.makedirs(local_driver_dir, exist_ok=True)

    print("📥 Step 1: Downloading FinBERT to Driver's LOCAL /tmp/ first...")
    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

    tokenizer.save_pretrained(local_driver_dir)
    model.save_pretrained(local_driver_dir)
    print("✅ Local download finished without any I/O block.")

    print("📤 Step 2: Copying stable model files into permanent Serverless Volume...")
    os.makedirs(shared_model_dir, exist_ok=True)

    for filename in os.listdir(local_driver_dir):
        local_file_path = os.path.join(local_driver_dir, filename)
        volume_file_path = os.path.join(shared_model_dir, filename)
        shutil.copy2(local_file_path, volume_file_path)

    if not _model_is_complete(shared_model_dir):
        raise RuntimeError(f"❌ Model copy to Volume failed at: {shared_model_dir}.")
    print("🎉 Model successfully sync'd to permanent Serverless Volume!")
else:
    print("🎉 FinBERT already exists in the Shared UC Volume. Skipping download!")


def split_text_into_chunks(text, max_words=350):
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


# ==========================================
# 1. الـ SCHEMA الخاص بالـ mapInPandas
# ==========================================
output_schema = StructType([
    StructField("ticker", StringType(), True),
    StructField("date", DateType(), True),
    StructField("positive_score", DoubleType(), True),
    StructField("negative_score", DoubleType(), True),
    StructField("neutral_score", DoubleType(), True)
])


# ==========================================
# 2. دالة الـ MAP IN PANDAS
# ==========================================
def predict_multi_sentiment_map(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    import os
    from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
    device = 0 if torch.cuda.is_available() else -1
    local_path = os.path.abspath(model_udf_path)
    tokenizer = AutoTokenizer.from_pretrained(local_path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(local_path, local_files_only=True)

    classifier = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        device=device,
        batch_size=32, 
    )

    for pdf in iterator:
        results = []
        for index, row in pdf.iterrows():
            text_list = row["all_news_texts"]
            ticker = row["ticker"]
            date = row["date"]

            if text_list is None or (hasattr(text_list, 'size') and text_list.size == 0) or len(text_list) == 0:
                results.append((ticker, date, 0.0, 0.0, 0.0))
                continue

            pos_sum, neg_sum, neu_sum = 0.0, 0.0, 0.0
            valid_news_count = 0

            for text in text_list:
                if text and text.strip():
                    try:
                        chunks = split_text_into_chunks(text)
                        if not chunks:
                            continue
                            
                        chunk_results = classifier(chunks, truncation=True, max_length=512)
                        
                        chunk_pos, chunk_neg, chunk_neu = 0.0, 0.0, 0.0
                        for res in chunk_results:
                            label = res['label']
                            score = res['score']

                            if label == 'positive':
                                chunk_pos += score
                            elif label == 'negative':
                                chunk_neg += score
                            elif label == 'neutral':
                                chunk_neu += score

                        total_chunks = len(chunks)
                        pos_sum += (chunk_pos / total_chunks)
                        neg_sum += (chunk_neg / total_chunks)
                        neu_sum += (chunk_neu / total_chunks)
                        valid_news_count += 1

                    except Exception as e:
                        continue

            if valid_news_count > 0:
                results.append((
                    ticker,
                    date,
                    float(pos_sum / valid_news_count),
                    float(neg_sum / valid_news_count),
                    float(neu_sum / valid_news_count),
                ))
            else:
                results.append((ticker, date, 0.0, 0.0, 0.0))

        yield pd.DataFrame(results, columns=["ticker", "date", "positive_score", "negative_score", "neutral_score"])


# ==========================================
# 3. ENSURE COLUMNS EXIST & CHECKPOINT
# ==========================================
print("🛡️ Verifying 3D schema columns inside stock_prices_raw...")
for col_name in ["positive_score", "negative_score", "neutral_score"]:
    try:
        spark.sql(f"ALTER TABLE {prices_table_name} ADD COLUMNS ({col_name} DOUBLE)")
    except Exception as e:
        print(f"ℹ️ Column '{col_name}' already exists.")

full_news_df = spark.table(news_table_name) \
    .withColumn("date", F.to_date("published_date")) \
    .groupBy("ticker", "date") \
    .agg(F.collect_list("text_for_finbert").alias("all_news_texts"))

target_delta_table = DeltaTable.forName(spark, prices_table_name)

existing_progress_df = target_delta_table.toDF() \
    .filter("positive_score IS NOT NULL AND negative_score IS NOT NULL") \
    .select("ticker", F.to_date("date").alias("date"))

news_to_process_df = full_news_df.join(existing_progress_df, on=["ticker", "date"], how="left_anti")

remaining_records = news_to_process_df.count()
print(f"🚀 Optimized Checkpoint: Found {remaining_records} missing news records to calculate.")

# ==========================================
# 4. INFERENCE & TARGETED BATCH MERGE (The Monitored Loop)
# ==========================================
if remaining_records > 0:
    print("🧠 Starting Micro-Batching FinBERT inference to prevent freezing...")
    
    # معالجة دفعات بحجم مناسب جداً لسرعة الـ CPU والـ Workers للإنهاء السريع لـ الحلقات
    batch_size = 5000 
    
    while True:
        # أخذ دفعة محدودة جداً من البيانات المتبقية
        batch_df = news_to_process_df.limit(batch_size)
        current_batch_count = batch_df.count()
        
        if current_batch_count == 0:
            break
            
        print(f"📦 Processing micro-batch of {current_batch_count} records...")
        
        # توزيع الدفعة الصغيرة على عدد بارتيشنز متناسق لضمان عمل كافة الـ Workers معاً بالتوازي لإنهاء الـ 5000 سجل بسرعة
        sentiment_results_df = batch_df.repartition(20).mapInPandas(predict_multi_sentiment_map, schema=output_schema)
        
        print("💾 Writing batch to temp storage...")
        sentiment_results_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(temp_results_table)
            
        print("🔗 Merging batch into target table...")
        cached_updates = spark.table(temp_results_table)
        target_delta_table.alias("prices") \
            .merge(
                source=cached_updates.alias("updates"),
                condition="prices.ticker = updates.ticker AND to_date(prices.date) = updates.date"
            ) \
            .whenMatchedUpdate(set={
                "positive_score": "updates.positive_score",
                "negative_score": "updates.negative_score",
                "neutral_score": "updates.neutral_score"
            }) \
            .execute()
            
        print("✅ Micro-batch successfully processed and merged!")
        
        # تحديث المتبقي باستبعاد السجلات التي تم تحديثها لتوها
        existing_progress_df = target_delta_table.toDF() \
            .filter("positive_score IS NOT NULL AND negative_score IS NOT NULL") \
            .select("ticker", F.to_date("date").alias("date"))
            
        news_to_process_df = full_news_df.join(existing_progress_df, on=["ticker", "date"], how="left_anti")
        
        if news_to_process_df.count() == 0:
            print("🎉 All batches completed successfully!")
            break

    print("✅ Full 3D Sentiment pipeline completed via Micro-Batches!")
else:
    print("🎉 All 3 sentiment dimensions are already completely calculated and updated!")

# ==========================================
# 5. SAFE PREVIEW
# ==========================================
print("📊 Fetching a safe preview of updated data...")
final_ml_ready_df = target_delta_table.toDF().fillna({
    "positive_score": 0.0,
    "negative_score": 0.0,
    "neutral_score": 0.0
})

final_ml_ready_df.select("date", "ticker", "close", "positive_score", "negative_score", "neutral_score") \
    .filter("positive_score > 0") \
    .show(10, truncate=False)